In [1]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import joblib

from tensorflow.keras import layers, Model
from scipy.signal import resample
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
DATA_DIRS = ["../watch/src/gesture_data_filtered",
            "../watch/src/sofie_gesture_data_filtered_24",
            "../watch/src/sofie_gesture_data_filtered_28"]

WINDOW_SIZE = 128
STRIDE = 16
CHANNELS = ["ax", "ay", "az", "gx", "gy", "gz"]

GESTURE_CODES = [
    "D",
    "De",
    "Dn",
    "Ds",
    "DUD",
    "Dw",
    "N",
    "U",
    "Ue",
    "Un",
    "Uw",
]
label_map = {code: i for i, code in enumerate(GESTURE_CODES)}
reverse_label_map = {i: code for code, i in label_map.items()}

In [3]:
def get_gesture_code(path):
    filename = os.path.basename(path)

    if filename.startswith("CCW_"):
        return "CCW"
    if filename.startswith("CW_"):
        return "CW"

    return filename.split("_")[0]

def make_windows(data, window_size=WINDOW_SIZE, stride=STRIDE):
    windows = []

    for start in range(0, len(data) - window_size + 1, stride):
        end = start + window_size
        windows.append(data[start:end])

    return windows


def load_dataset():
    X = []
    y = []
    used_files = []
    skipped_short = 0

    files = []

    for data_dir in DATA_DIRS:
        found = glob.glob(os.path.join(data_dir, "*.csv"))
        print(data_dir, "->", len(found), "CSV files")
        files.extend(found)

    print("Total CSV files found:", len(files))

    for path in files:
        gesture_code = get_gesture_code(path)

        if gesture_code not in label_map:
            continue

        df = pd.read_csv(path)

        if not all(ch in df.columns for ch in CHANNELS):
            print(f"Skipping {path}: missing IMU columns")
            continue

        data = df[CHANNELS].values.astype(np.float32)

        if len(data) < WINDOW_SIZE:
            skipped_short += 1
            continue

        windows = make_windows(data)

        for window in windows:
            X.append(window)
            y.append(label_map[gesture_code])
            used_files.append(path)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)

    print("Skipped short files:", skipped_short)

    return X, y, used_files

In [4]:
# Test if the data is put through for CW/CCW
X, y, used_files = load_dataset()

print("X shape:", X.shape)
print("y shape:", y.shape)

for gesture, idx in label_map.items():
    print(gesture, np.sum(y == idx))


print("Number of classes:", len(label_map))
print(label_map)
print("X shape:", X.shape)

../watch/src/gesture_data_filtered -> 1127 CSV files
../watch/src/sofie_gesture_data_filtered_24 -> 976 CSV files
../watch/src/sofie_gesture_data_filtered_28 -> 247 CSV files
Total CSV files found: 2350
Skipped short files: 841
X shape: (9032, 128, 6)
y shape: (9032,)
D 429
De 743
Dn 600
Ds 561
DUD 1083
Dw 641
N 1129
U 1386
Ue 917
Un 454
Uw 1089
Number of classes: 11
{'D': 0, 'De': 1, 'Dn': 2, 'Ds': 3, 'DUD': 4, 'Dw': 5, 'N': 6, 'U': 7, 'Ue': 8, 'Un': 9, 'Uw': 10}
X shape: (9032, 128, 6)


In [ ]:
# Checking which folder the skipped files came from (chatgpt)
from collections import defaultdict
import os
import glob
import pandas as pd

DATA_DIRS = [
    "../watch/src/gesture_data_filtered",
    "../watch/src/sofie_gesture_data_filtered_24",
    "../watch/src/sofie_gesture_data_filtered_28",
]

stats = defaultdict(lambda: {"total": 0, "used": 0, "skipped": 0})

for data_dir in DATA_DIRS:
    files_in_dir = glob.glob(os.path.join(data_dir, "*.csv"))

    dataset = os.path.basename(data_dir)

    for path in files_in_dir:
        stats[dataset]["total"] += 1

        df = pd.read_csv(path)

        if len(df) < WINDOW_SIZE:
            stats[dataset]["skipped"] += 1
        else:
            stats[dataset]["used"] += 1

print(f"Window size = {WINDOW_SIZE}\n")

for dataset, s in stats.items():
    pct = 100 * s["skipped"] / s["total"] if s["total"] else 0

    print(dataset)
    print(f"  Total files   : {s['total']}")
    print(f"  Used files    : {s['used']}")
    print(f"  Skipped files : {s['skipped']}")
    print(f"  Percent skip  : {pct:.1f}%")
    print()

Window size = 128

gesture_data_filtered
  Total files   : 1127
  Used files    : 959
  Skipped files : 168
  Percent skip  : 14.9%

sofie_gesture_data_filtered_24
  Total files   : 976
  Used files    : 176
  Skipped files : 800
  Percent skip  : 82.0%

sofie_gesture_data_filtered_28
  Total files   : 247
  Used files    : 182
  Skipped files : 65
  Percent skip  : 26.3%



In [6]:
# Checking to make sure all csv files got imported
import os
import glob

DATA_DIRS = ["../watch/src/gesture_data_filtered",
            "../watch/src/sofie_gesture_data_filtered_24",
            "../watch/src/sofie_gesture_data_filtered_28"]

all_files = []

for data_dir in DATA_DIRS:
    files = glob.glob(os.path.join(data_dir, "*.csv"))

    print(data_dir)
    print("CSV files found:", len(files))

    if len(files) > 0:
        print("First few files:")
        for f in files[:5]:
            print(" ", f)

    print()
    all_files.extend(files)

print("Total CSV files found:", len(all_files))

../watch/src/gesture_data_filtered
CSV files found: 1127
First few files:
  ../watch/src/gesture_data_filtered\De_no_tremor_1.csv
  ../watch/src/gesture_data_filtered\De_no_tremor_10.csv
  ../watch/src/gesture_data_filtered\De_no_tremor_11.csv
  ../watch/src/gesture_data_filtered\De_no_tremor_12.csv
  ../watch/src/gesture_data_filtered\De_no_tremor_13.csv

../watch/src/sofie_gesture_data_filtered_24
CSV files found: 976
First few files:
  ../watch/src/sofie_gesture_data_filtered_24\CCW_no_tremor_1.csv
  ../watch/src/sofie_gesture_data_filtered_24\CCW_no_tremor_10.csv
  ../watch/src/sofie_gesture_data_filtered_24\CCW_no_tremor_100.csv
  ../watch/src/sofie_gesture_data_filtered_24\CCW_no_tremor_11.csv
  ../watch/src/sofie_gesture_data_filtered_24\CCW_no_tremor_12.csv

../watch/src/sofie_gesture_data_filtered_28
CSV files found: 247
First few files:
  ../watch/src/sofie_gesture_data_filtered_28\D_no_tremor_1.csv
  ../watch/src/sofie_gesture_data_filtered_28\D_no_tremor_10.csv
  ../watch/s

In [7]:
# TRYNA DEBUG

import os
import pandas as pd
import numpy as np

for f in files[:20]:
    print(os.path.basename(f))


for f in files:
    name = os.path.basename(f)
    if "CW" in name or "CCW" in name:
        print(name)


for code in ["CW", "CCW"]:
    matches = [
        f for f in files
        if os.path.basename(f).startswith(code + "_")
    ]

    lengths = []

    for f in matches:
        df = pd.read_csv(f)
        lengths.append(len(df))

    print(code)
    print("files:", len(matches))

    if len(lengths) == 0:
        print("No matching files found.")
        print()
        continue

    print("min:", min(lengths))
    print("avg:", np.mean(lengths))
    print("max:", max(lengths))
    print(">=128:", sum(l >= 128 for l in lengths))
    print()

D_no_tremor_1.csv
D_no_tremor_10.csv
D_no_tremor_100.csv
D_no_tremor_101.csv
D_no_tremor_102.csv
D_no_tremor_11.csv
D_no_tremor_12.csv
D_no_tremor_13.csv
D_no_tremor_14.csv
D_no_tremor_15.csv
D_no_tremor_16.csv
D_no_tremor_17.csv
D_no_tremor_18.csv
D_no_tremor_19.csv
D_no_tremor_2.csv
D_no_tremor_20.csv
D_no_tremor_21.csv
D_no_tremor_22.csv
D_no_tremor_23.csv
D_no_tremor_24.csv
CW
files: 0
No matching files found.

CCW
files: 0
No matching files found.



In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


# Normalize data
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, 6))
X_train = scaler.transform(X_train.reshape(-1, 6)).reshape(X_train.shape)
X_val = scaler.transform(X_val.reshape(-1, 6)).reshape(X_val.shape)
X_test = scaler.transform(X_test.reshape(-1, 6)).reshape(X_test.shape)

Train: (6322, 128, 6)
Val: (1355, 128, 6)
Test: (1355, 128, 6)


In [9]:
def build_model(num_classes):
    inputs = tf.keras.Input(shape=(WINDOW_SIZE, 6))

    x = layers.Conv1D(64, 5, padding="causal", activation="relu")(inputs)
    x = layers.Conv1D(128, 5, padding="causal", activation="relu")(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, 3, padding="causal", activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    x = layers.GRU(128, return_sequences=False)(x)

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs, outputs)


model = build_model(num_classes=len(GESTURE_CODES))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 6)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 128, 64)        │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 128, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 64, 128)        │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │           715 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 200,395 (782.79 KB)

 Trainable params: 200,395 (782.79 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=16
)

Epoch 1/40
396/396 ━━━━━━━━━━━━━━━━━━━━ 25s 56ms/step - accuracy: 0.7037 - loss: 0.8862 - val_accuracy: 0.9225 - val_loss: 0.2390
Epoch 2/40
396/396 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.9152 - loss: 0.2670 - val_accuracy: 0.9314 - val_loss: 0.2148
Epoch 3/40
396/396 ━━━━━━━━━━━━━━━━━━━━ 22s 55ms/step - accuracy: 0.9378 - loss: 0.1900 - val_accuracy: 0.9491 - val_loss: 0.1663
Epoch 4/40
396/396 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.9562 - loss: 0.1336 - val_accuracy: 0.9587 - val_loss: 0.1150
Epoch 5/40
396/396 ━━━━━━━━━━━━━━━━━━━━ 22s 56ms/step - accuracy: 0.9587 - loss: 0.1349 - val_accuracy: 0.9587 - val_loss: 0.1193
Epoch 6/40
396/396 ━━━━━━━━━━━━━━━━━━━━ 20s 51ms/step - accuracy: 0.9655 - loss: 0.1076 - val_accuracy: 0.9697 - val_loss: 0.0943
Epoch 7/40
396/396 ━━━━━━━━━━━━━━━━━━━━ 22s 56ms/step - accuracy: 0.9709 - loss: 0.0926 - val_accuracy: 0.9624 - val_loss: 0.0964
Epoch 8/40
396/396 ━━━━━━━━━━━━━━━━━━━━ 24s 60ms/step - accuracy: 0.9712 - loss: 0.0850 - 

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_test, y_pred, target_names=GESTURE_CODES))
print(confusion_matrix(y_test, y_pred))

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Gesture Classification Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Gesture Classification Loss")
plt.legend()
plt.grid(True)
plt.show()

cm_norm = confusion_matrix(y_test, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(9, 9))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=GESTURE_CODES
)
disp.plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=True)
plt.title("Normalized Gesture Confusion Matrix")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# model.save("gesture_cnn_gru_sliding_192.keras")
# joblib.dump(scaler, "gesture_scaler_sliding_192.pkl")
# print("Saved model as gesture_cnn_gru_resampled.keras")

In [ ]:
import tensorflow as tf
import keras

print(tf.__version__)
print(keras.__version__)

In [ ]:
# Check to make sure an even amount of data per class
for gesture, idx in label_map.items():
    print(gesture, np.sum(y == idx))

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(test_acc)

print(classification_report(y_test, y_pred, target_names=GESTURE_CODES))
print(confusion_matrix(y_test, y_pred))